# Task 2 — Data Preprocessing for a Regression Problem
### Uber Fare Prediction | Follow-up to Task 1 (EDA)

This notebook turns the raw `final_internship_data.csv` into a clean, encoded, scaled,
**leak-free** dataset ready for regression, and trains a baseline model plus one
hyperparameter-tuned model.

Every decision below is justified using what Task 1's EDA actually found — not a fixed
recipe. The key EDA findings that drive these decisions:

| EDA finding (Task 1) | Preprocessing consequence |
|---|---|
| `fare_amount`: mean \$11.35 vs. median \$8.50, max \$200 | Target is right-skewed → **log1p transform** before training |
| `distance`: median 2.2 km but max 8,670 km | Physically impossible for an NYC trip → **data error, not a rare-true outlier** → winsorize |
| `passenger_count`, `Car Condition`, `Traffic Condition` all had correlation ≤ 0.02 with fare | Kept as features (tree models can still find interactions) but **not hand-dropped on EDA correlation alone** — left to embedded feature selection instead |
| `hour` showed a non-linear, wrap-around pattern (demand peaks 7 PM, avg. fare peaks 5 AM) | Treated as **cyclical** (`sin`/`cos`), not a plain integer |
| `User ID`, `User Name`, `Driver Name`, `key` | Identifiers, explicitly called out as *"not useful for modeling"* in the data dictionary → dropped |

**Pipeline overview:** raw CSV → deterministic cleaning (drop ID columns, remove exact/key
duplicates, drop rows with minimal missingness) → train/test split → `sklearn.Pipeline`
(winsorize → encode → scale → cyclical-encode, all fit on train only) → `TransformedTargetRegressor`
(log1p/expm1) → 5-fold CV baseline → `RandomizedSearchCV` tuned model → evaluation on the
untouched test set.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')

try:
    df_raw = pd.read_csv('final_internship_data.csv')
except FileNotFoundError:
    df_raw = pd.read_csv(r'/content/final_internship_data.csv')

print('Raw shape:', df_raw.shape)
df_raw.head(3)


## 1. Quantify Missingness

Per the task instructions: quantify missingness first, then decide *drop vs. impute vs. flag*
based on the pattern — the pattern should drive the method, not habit.

We compute `isnull().mean()` per column. From Task 1 we already know the overall row loss
from combined missing + invalid values was small (58,365 → 56,667, ≈ 2.9%), which is consistent
with **MCAR (Missing Completely At Random)** — there's no reason to believe a trip's fare is
more likely to be missing *because* of its own weather or traffic value. With MCAR and small
missingness (<3%), the safe default is `dropna()` rather than imputation, which would add
little value and some noise for such a small fraction of rows.


In [ ]:
missingness = df_raw.isnull().mean().sort_values(ascending=False)
print(missingness[missingness > 0])
print('\nColumns with >0% missing:', (missingness > 0).sum())
print('Max missingness in any single column: {:.2%}'.format(missingness.max() if len(missingness) else 0))


**Decision:** every column with missing values here has missingness well under the 2–3%
threshold (per Task 1's EDA, overall row loss was ~2.9% combined with invalid values). We drop
rows with missing values in modeling-relevant columns with `dropna()`, rather than imputing —
imputation exists to preserve rows when missingness is large or structured; at this scale it adds
complexity without meaningfully improving the dataset. We don't drop any *column* outright, since
none is anywhere near the 50–60% missing threshold that would justify that.


In [ ]:
# Drop columns explicitly called out as "not useful for modeling" in the data dictionary,
# plus the trip-level key (kept only for de-duplication in Section 2, dropped afterward).
ID_COLS = ['User ID', 'User Name', 'Driver Name']

df = df_raw.drop(columns=ID_COLS)

MODEL_COLS = [c for c in df.columns if c != 'key']
df = df.dropna(subset=MODEL_COLS)
print('Shape after dropping ID columns and missing rows:', df.shape)


## 2. Detecting and Removing Duplicates

- **Exact duplicates** (identical across all columns): safe to drop outright.
- **Key-based duplicates**: the data dictionary states `key` is *"Unique trip ID — use only for
  deduplication"*. If the same `key` appears twice, it's a scraping/export duplicate of the same
  trip, not two different trips — we keep the first occurrence and drop the rest.

We check both, in that order, before dropping the `key` column itself (it carries no
predictive signal once de-duplication is done).


In [ ]:
exact_dupes = df.duplicated().sum()
key_dupes = df.duplicated(subset=['key']).sum()
print(f'Exact duplicate rows: {exact_dupes}')
print(f'Key-based duplicate rows (same trip key): {key_dupes}')

df = df.drop_duplicates()
df = df.drop_duplicates(subset=['key'], keep='first')
df = df.drop(columns=['key'])
print('Shape after de-duplication:', df.shape)


## 3. Invalid Values vs. Genuine Outliers — a distinction Task 1 blurred on purpose

Task 1's cleaning treated "`fare_amount <= 0`" and "`distance <= 0`" as the same category as
missing data — and that's correct: a $0 fare or 0 km trip is a **data-entry error**, the
same way a sensor reading of ‑999°C is a malfunction, not a rare-but-real cold day (per this
task's own guidance). We replicate that rule here as part of deterministic cleaning, since
it doesn't depend on any statistic that could leak from test to train.

The **large-distance outliers** (max 8,670 km, median 2.2 km) are a different case, and we treat
them differently on purpose:
- A $0 fare is *impossible* regardless of context → always an error → drop/convert to NaN.
- An 8,670 km "NYC trip" is *also* impossible (NYC's longest plausible in-city trip is well
  under 100 km) → still an error, most likely corrupted GPS/geocoding, not a rare valid long
  trip (like the mansion-price example in this task's notes). But rather than drop these rows
  outright and lose other valid signal in them (fare, weather, hour, etc.), we **cap
  (winsorize)** the distance value at a learned percentile. Because that percentile is a
  *statistic*, it must be learned from the training set only — so this step moves into the
  `sklearn.Pipeline` in Section 6, not here.


In [ ]:
# Invalid-value rule (data-entry errors, not outliers): identical logic to Task 1.
before = df.shape[0]
df = df[(df['fare_amount'] > 0) & (df['distance'] > 0)].copy()
print(f'Dropped {before - df.shape[0]} rows with non-positive fare or distance (data errors).')
print('Shape after invalid-value cleaning:', df.shape)


## 4. Feature Engineering

These transformations are **deterministic** (they don't learn any statistic from the data), so
applying them before the split introduces no leakage — the same formula would produce the same
result whether computed on the full dataset, the train set, or a single new row at inference time.

- **`is_weekend`** from `weekday` — Task 1's extra analysis checked weekend vs. weekday fare
  directly ($11.36 vs. $11.31 — barely different) but `is_weekend` is still worth keeping as a
  feature for the model to weigh itself, rather than deciding its importance from a single
  univariate comparison.
- **Cyclical hour (`hour_sin`, `hour_cos`)** — Task 1's EDA found demand peaking near 7 PM and
  average fare peaking near 5 AM, i.e. the relationship between `hour` and the target is
  *non-linear and wraps around* (hour 23 and hour 0 are one hour apart, not 23). A raw integer or
  even one-hot-encoding 24 columns would hide that adjacency. `sin`/`cos` encoding preserves it
  with just two columns.
- **Drop `pickup_datetime`** — already fully decomposed into `hour`, `day`, `month`, `weekday`,
  `year`; keeping the raw string adds nothing a model can use directly.
- **Drop raw pickup/dropoff latitude & longitude** — `distance`, `bearing`, and the four
  airport/landmark distances (`jfk_dist`, `ewr_dist`, `lga_dist`, `sol_dist`) already encode the
  spatial information those four raw coordinates carry, at lower dimensionality and without the
  severe multicollinearity four correlated coordinates would introduce for a linear model.


In [ ]:
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

DROP_COLS = ['pickup_datetime', 'pickup_longitude', 'pickup_latitude',
             'dropoff_longitude', 'dropoff_latitude', 'hour', 'weekday']
df = df.drop(columns=DROP_COLS)

print('Columns going into the model:')
print(df.columns.tolist())
df.head(3)


## 5. Target Distribution — Why `fare_amount` Needs a Log Transform

Task 1 flagged this directly: mean \$11.35 vs. median \$8.50, with a long right tail up to \$200.
Training directly on a heavily right-skewed target lets a handful of expensive trips dominate the
squared-error loss. We confirm the skew visually here, then apply `np.log1p` to the target via
`TransformedTargetRegressor` in Section 8 — fit entirely inside the model wrapper, so every
metric we report in Section 9 is automatically converted back with `np.expm1` into real dollars,
never left on the log scale.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(df['fare_amount'], bins=60, color='steelblue')
axes[0].set_title('fare_amount — raw')
axes[0].set_xlabel('Fare ($)')

axes[1].hist(np.log1p(df['fare_amount']), bins=60, color='steelblue')
axes[1].set_title('log1p(fare_amount)')
axes[1].set_xlabel('log(1 + Fare)')
plt.tight_layout()
plt.show()

print('Skew (raw):', df['fare_amount'].skew().round(2))
print('Skew (log1p):', np.log1p(df['fare_amount']).skew().round(2))


## 6. Train/Test Split — Before Any Statistic Is Learned

Every remaining step (winsorizing thresholds, encoder categories, scaler mean/median) is a
statistic learned from data, so the split happens **now**, before any of them are fit. This
dataset is treated as i.i.d. — trip rows aren't a continuous time series we're forecasting
forward — so a random 80/20 split is appropriate. (If this were instead a *demand-forecasting*
task, we'd switch to a chronological split using the existing `year`/`month`/`day` columns,
exactly as the task notes recommend for time-ordered data.)


In [ ]:
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)


## 7. Encoding, Scaling & Outlier Capping — All Inside One Leak-Free Pipeline

Three technique choices, each justified by an EDA finding:

| Column(s) | Technique | Why |
|---|---|---|
| `Car Condition` | **Ordinal encoding** (`Bad < Good < Very Good < Excellent`) | Task 1's boxplot showed near-identical medians across levels, but the levels *do* have a real quality order — ordinal preserves that order in one column instead of inventing false distance with one-hot |
| `Weather`, `Traffic Condition` | **One-hot encoding** | Low cardinality (5 and 3 categories), no natural order — one-hot is the textbook fit, and won't explode dimensionality at this cardinality |
| `distance`, `jfk_dist`, `ewr_dist`, `lga_dist`, `sol_dist`, `nyc_dist`, `passenger_count` | **Winsorize (cap at 1st/99th percentile) → `RobustScaler`** | Task 1 showed `distance` has extreme outliers (max 8,670 km); `RobustScaler` uses median/IQR instead of mean/std, so it isn't itself distorted by whatever remains after capping |
| `hour_sin`, `hour_cos`, `is_weekend` | **Passthrough, no scaling** | Already bounded in [-1, 1] / {0, 1} — scaling a bounded or binary column just adds noise, per the task's own senior tip about not scaling one-hot columns |

The winsorizing thresholds are learned with `fit()` on `X_train` only, inside a custom
`sklearn`-compatible transformer, so the exact same train-derived caps are applied to
`X_test` with `transform()` — no statistic ever crosses from test into train.


In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """Caps each numeric column at [lower_q, upper_q] percentiles learned on fit()."""

    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_ = X.quantile(self.lower_q)
        self.upper_ = X.quantile(self.upper_q)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.clip(lower=self.lower_, upper=self.upper_, axis=1).values


NUMERIC_OUTLIER_COLS = ['distance', 'jfk_dist', 'ewr_dist', 'lga_dist',
                         'sol_dist', 'nyc_dist', 'passenger_count', 'bearing']
ORDINAL_COL = ['Car Condition']
ONEHOT_COLS = ['Weather', 'Traffic Condition']
PASSTHROUGH_COLS = ['hour_sin', 'hour_cos', 'is_weekend', 'day', 'month', 'year']

CAR_CONDITION_ORDER = [['Bad', 'Good', 'Very Good', 'Excellent']]

numeric_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),   # safety net; rows were already dropna'd
    ('winsorize', Winsorizer(lower_q=0.01, upper_q=0.99)),
    ('scale', RobustScaler()),
])

ordinal_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=CAR_CONDITION_ORDER)),
])

onehot_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipe, NUMERIC_OUTLIER_COLS),
    ('ord', ordinal_pipe, ORDINAL_COL),
    ('ohe', onehot_pipe, ONEHOT_COLS),
    ('pass', 'passthrough', PASSTHROUGH_COLS),
])

preprocessor


## 8. Baseline Model — Ridge Regression with 5-Fold Cross-Validation

A linear model is the right baseline here specifically *because* it's sensitive to scale and
encoding choice — it will immediately expose whether Section 7's preprocessing was done
correctly, before we move to a model family (tree-based) that would quietly tolerate mistakes.

The full `preprocessor` is wrapped inside the same `Pipeline` passed to `cross_validate`, and the
target is log-transformed via `TransformedTargetRegressor` — so each of the 5 folds refits
imputation, winsorizing thresholds, encoders and scaler **independently on its own training
portion**, exactly as the task's leakage warning requires.


In [ ]:
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', TransformedTargetRegressor(
        regressor=Ridge(alpha=1.0, random_state=RANDOM_STATE),
        func=np.log1p,
        inverse_func=np.expm1,
    )),
])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2',
}

baseline_scores = cross_validate(
    baseline_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1
)

print('Baseline Ridge — 5-fold CV (train set only):')
print(f"  RMSE: {-baseline_scores['test_rmse'].mean():.3f} (+/- {baseline_scores['test_rmse'].std():.3f})")
print(f"  MAE:  {-baseline_scores['test_mae'].mean():.3f} (+/- {baseline_scores['test_mae'].std():.3f})")
print(f"  R2:   {baseline_scores['test_r2'].mean():.3f} (+/- {baseline_scores['test_r2'].std():.3f})")


## 9. Hyperparameter-Tuned Model — Random Forest + `RandomizedSearchCV`

Switching to `RandomForestRegressor`: tree-based models don't need the `RobustScaler`/encoding
choices to be perfect to perform well (they split on thresholds, not weighted sums), but we keep
the *same* `preprocessor` pipeline regardless — a Random Forest still benefits from the winsorized
distance column (fewer meaningless deep splits chasing 8,670 km outliers) and still needs
categoricals in numeric form.

`RandomizedSearchCV` (not a full grid) is the right tool here per the task's own
guidance table: `n_estimators` × `max_depth` × `min_samples_split` × `max_features` is a large
combinatorial space, and sampling a fixed number of random combinations gets most of the benefit
of an exhaustive grid at a fraction of the compute.


In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', TransformedTargetRegressor(
        regressor=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        func=np.log1p,
        inverse_func=np.expm1,
    )),
])

param_distributions = {
    'model__regressor__n_estimators': [100, 200, 300, 400],
    'model__regressor__max_depth': [None, 6, 10, 16, 24],
    'model__regressor__min_samples_split': [2, 5, 10],
    'model__regressor__max_features': ['sqrt', 'log2', 0.5, 1.0],
}

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print('Best CV RMSE (train set):', -search.best_score_)
print('Best params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

best_model = search.best_estimator_


## 10. Final Evaluation on the Untouched Test Set

`X_test` / `y_test` have not been seen by `fit()`, `cross_validate()`, or `RandomizedSearchCV`
at any point up to here — this is the one and only time they're used, exactly as the task's
senior tip on tuning warns. Metrics are computed in original dollars (the `TransformedTargetRegressor`
already applied `np.expm1` internally), not on the log scale.


In [ ]:
y_pred_baseline = baseline_pipeline.fit(X_train, y_train).predict(X_test)
y_pred_tuned = best_model.predict(X_test)

def report(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{name:>22} |  RMSE: {rmse:6.3f}   MAE: {mae:6.3f}   R2: {r2:6.3f}')

print('Held-out test set performance (original $ scale):')
report('Baseline (Ridge)', y_test, y_pred_baseline)
report('Tuned (Random Forest)', y_test, y_pred_tuned)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_tuned, alpha=0.3, s=12, color='steelblue')
lims = [0, max(y_test.max(), y_pred_tuned.max())]
ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Fare ($)')
ax.set_ylabel('Predicted Fare ($)')
ax.set_title('Tuned Random Forest — Predicted vs. Actual (test set)')
ax.legend()
plt.tight_layout()
plt.show()


## 11. Embedded Feature Selection Check

Task 1's correlations suggested `passenger_count`, `Car Condition`, and `Traffic Condition` carry
little *linear* signal. Rather than drop them by hand on that basis — a single Pearson correlation
can miss a feature's value in combination with others, which a tree model can exploit — we check
the tuned Random Forest's own `feature_importances_` as an embedded, model-driven confirmation
(or correction) of that EDA hypothesis.


In [ ]:
ohe_feature_names = (
    best_model.named_steps['preprocessor']
    .named_transformers_['ohe']
    .named_steps['encode']
    .get_feature_names_out(ONEHOT_COLS)
)
all_feature_names = (
    NUMERIC_OUTLIER_COLS + ORDINAL_COL + list(ohe_feature_names) + PASSTHROUGH_COLS
)

importances = best_model.named_steps['model'].regressor_.feature_importances_
importance_df = (
    pd.DataFrame({'feature': all_feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
print(importance_df.to_string(index=False))

plt.figure(figsize=(9, 6))
plt.barh(importance_df['feature'][:12][::-1], importance_df['importance'][:12][::-1], color='steelblue')
plt.title('Top 12 Feature Importances — Tuned Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## Conclusion — What Changed, and Why

| Step | Decision | Driven by |
|---|---|---|
| Missing values | `dropna()` on modeling columns | Missingness <3%, consistent with MCAR (Task 1) |
| Duplicates | Dropped exact + key-based duplicates | `key` explicitly documented as dedup-only |
| Invalid values | Dropped non-positive `fare_amount` / `distance` | Same logic as a malfunctioning sensor reading — not a real trip |
| Distance outliers | Winsorized at [1st, 99th] percentile, **fit on train only** | Task 1: max 8,670 km vs. 2.2 km median — physically impossible, but informative rows otherwise |
| `Car Condition` | Ordinal encoding | Real quality order exists, even though Task 1 found it barely moves fare |
| `Weather`, `Traffic Condition` | One-hot encoding | Low cardinality, no order |
| `hour` | Dropped in favor of `hour_sin`/`hour_cos` | Task 1 found a wrap-around, non-linear time pattern |
| Numeric distance features | `RobustScaler` | Robust to the outliers that remain after capping |
| Target `fare_amount` | `log1p` via `TransformedTargetRegressor` | Right-skewed (mean > median), confirmed by skew statistic |
| Split | 80/20 random, before any `fit()` | i.i.d. trip data, no leakage |
| CV | 5-fold `KFold` | Standard for i.i.d. tabular regression |
| Tuning | `RandomizedSearchCV`, 25 iterations | Large hyperparameter space, same CV scheme used for reporting |

**Next step:** the tuned Random Forest can be compared against gradient boosting
(XGBoost/LightGBM) using the exact same `preprocessor` and CV scheme, and the feature-importance
check in Section 11 is a natural starting point for a second, more aggressive feature-selection
pass (e.g. `SelectFromModel` wrapped inside the same pipeline, per the task's warning against
selecting features outside the CV loop).
